In [ ]:
import random
import numpy as np

In [ ]:
# 树鼩染色体长度数据
tree_shrew_lengths = {
    'chr1': 224450918, 'chr2': 187992121, 'chrX': 118708346, 'chr3': 137207235,
    'chr4': 121549664, 'chr5': 120916967, 'chr6': 117407209, 'chr7': 108272512,
    'chr8': 108089109, 'chr9': 104643080, 'chr10': 101339252, 'chr11': 97538608,
    'chr12': 94037081, 'chr13': 92324411, 'chr14': 89550385, 'chr15': 71760103,
    'chr16': 69746097, 'chr17': 66958136, 'chr18': 63467664, 'chr19': 57328337,
    'chr20': 54555981, 'chr21': 49773139, 'chr22': 52051345, 'chr23': 43822853,
    'chr24': 42262476, 'chr25': 42000901, 'chr26': 30576274, 'chr27': 25819293,
    'chr28': 26508867, 'chr29': 22611674, 'chr30': 21675798
}

In [ ]:
def create_optimized_tree_shrew_folds_v2(max_iterations=1000, seed=42):
    """创建优化的树鼩5-fold交叉验证，确保所有染色体都在测试集中出现"""
    
    total_bases = sum(tree_shrew_lengths.values())
    target_train = total_bases * 0.7  # 70%训练集
    target_test = total_bases * 0.2   # 20%测试集
    target_valid = total_bases * 0.1  # 10%验证集
    
    print(f"总碱基数: {total_bases:,} bp")
    print(f"目标比例 - 训练集70%: {target_train:,.0f} bp")
    print(f"          测试集20%: {target_test:,.0f} bp") 
    print(f"          验证集10%: {target_valid:,.0f} bp")
    print()
    
    all_chroms = list(tree_shrew_lengths.keys())
    
    best_folds = None
    best_score = float('inf')
    
    for iteration in range(max_iterations):
        # 每次使用不同的随机种子
        random.seed(seed + iteration)
        np.random.seed(seed + iteration)
        
        # 策略1: 确保每个染色体至少在1个fold的测试集中出现
        # 首先为每个染色体分配一个fold作为测试集
        chrom_to_test_fold = {}
        folds_test_chroms = {i: [] for i in range(1, 6)}
        
        # 随机分配每个染色体到一个fold的测试集
        shuffled_chroms = all_chroms.copy()
        random.shuffle(shuffled_chroms)
        
        for i, chrom in enumerate(shuffled_chroms):
            fold_num = (i % 5) + 1  # 循环分配到5个fold
            folds_test_chroms[fold_num].append(chrom)
            chrom_to_test_fold[chrom] = fold_num
        
        folds = {}
        
        for fold_num in range(1, 6):
            # 当前fold的测试染色体
            test_chroms = folds_test_chroms[fold_num].copy()
            
            # 剩余的染色体用于训练和验证
            remaining_chroms = [chrom for chrom in all_chroms if chrom not in test_chroms]
            random.shuffle(remaining_chroms)
            
            valid_chroms = []
            train_chroms = []
            
            test_bases = sum(tree_shrew_lengths[chrom] for chrom in test_chroms)
            valid_bases = 0
            train_bases = 0
            
            # 分配验证集
            for chrom in remaining_chroms:
                chrom_length = tree_shrew_lengths[chrom]
                if valid_bases < target_valid * 0.9:
                    valid_chroms.append(chrom)
                    valid_bases += chrom_length
                else:
                    train_chroms.append(chrom)
                    train_bases += chrom_length
            
            # 微调比例
            current_total = test_bases + valid_bases + train_bases
            
            # 调整验证集比例
            if valid_bases / current_total < 0.08:
                # 从训练集移动染色体到验证集
                for chrom in train_chroms.copy():
                    chrom_length = tree_shrew_lengths[chrom]
                    new_valid_ratio = (valid_bases + chrom_length) / (current_total)
                    if new_valid_ratio <= 0.12:
                        train_chroms.remove(chrom)
                        valid_chroms.append(chrom)
                        valid_bases += chrom_length
                        train_bases -= chrom_length
                        break
            
            elif valid_bases / current_total > 0.12:
                # 从验证集移动染色体到训练集
                for chrom in valid_chroms.copy():
                    chrom_length = tree_shrew_lengths[chrom]
                    new_valid_ratio = (valid_bases - chrom_length) / (current_total)
                    if new_valid_ratio >= 0.08:
                        valid_chroms.remove(chrom)
                        train_chroms.append(chrom)
                        valid_bases -= chrom_length
                        train_bases += chrom_length
                        break
            
            folds[f'fold{fold_num}'] = {
                'test': sorted(test_chroms),
                'valid': sorted(valid_chroms),
                'train': sorted(train_chroms)
            }
        
        # 验证所有染色体是否都在测试集中出现
        test_chroms_used = set()
        for fold_data in folds.values():
            test_chroms_used.update(fold_data['test'])
        
        coverage_score = len(test_chroms_used)
        
        # 计算比例偏差
        ratio_deviation = 0
        for fold_name, fold_data in folds.items():
            test_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['test'])
            valid_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['valid'])
            train_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['train'])
            total_bases = test_bases + valid_bases + train_bases
            
            test_ratio = test_bases / total_bases
            valid_ratio = valid_bases / total_bases
            train_ratio = train_bases / total_bases
            
            ratio_deviation += abs(test_ratio - 0.2) + abs(valid_ratio - 0.1) + abs(train_ratio - 0.7)
        
        # 综合评分
        if coverage_score == len(all_chroms):
            current_score = ratio_deviation
        else:
            current_score = ratio_deviation + (len(all_chroms) - coverage_score) * 100  # 加大惩罚
        
        # 更新最佳结果
        if current_score < best_score:
            best_score = current_score
            best_folds = folds.copy()
            best_coverage = coverage_score
        
        # 如果找到完美解，提前退出
        if coverage_score == len(all_chroms) and ratio_deviation < 0.1:
            print(f"在第 {iteration + 1} 次迭代找到满意解!")
            break
        
        if iteration % 100 == 0:
            print(f"迭代 {iteration}: 覆盖 {coverage_score}/{len(all_chroms)} 染色体, 比例偏差 {ratio_deviation:.4f}")
    
    print(f"\n最终结果: 覆盖 {best_coverage}/{len(all_chroms)} 染色体, 最佳评分 {best_score:.4f}")
    
    # 打印详细信息
    print("\n各fold分布详情:")
    for fold_name, fold_data in best_folds.items():
        test_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['test'])
        valid_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['valid'])
        train_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['train'])
        total_bases = test_bases + valid_bases + train_bases
        
        print(f"{fold_name}:")
        print(f"  测试集: {len(fold_data['test'])}条染色体, {test_bases:,}bp ({test_bases/total_bases*100:.1f}%)")
        print(f"  验证集: {len(fold_data['valid'])}条染色体, {valid_bases:,}bp ({valid_bases/total_bases*100:.1f}%)")
        print(f"  训练集: {len(fold_data['train'])}条染色体, {train_bases:,}bp ({train_bases/total_bases*100:.1f}%)")
    
    # 检查染色体覆盖情况
    all_test_chroms = set()
    for fold_data in best_folds.values():
        all_test_chroms.update(fold_data['test'])
    
    print(f"\n染色体覆盖检查: {len(all_test_chroms)}/{len(all_chroms)} 条染色体在测试集中出现")
    if len(all_test_chroms) < len(all_chroms):
        missing_chroms = set(all_chroms) - all_test_chroms
        print(f"缺失的染色体: {sorted(missing_chroms)}")
    
    return best_folds

In [ ]:
def calculate_and_validate_folds(folds):
    """计算并验证fold的碱基比例"""
    
    print("=== 树鼩基因组随机5-Fold划分验证 (7:2:1) ===\n")
    
    all_results = {}
    chrom_usage = {chrom: {'test': 0, 'valid': 0, 'train': 0} for chrom in tree_shrew_lengths.keys()}
    
    for fold_name, fold_data in folds.items():
        # 计算碱基数
        test_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['test'])
        valid_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['valid'])
        train_bases = sum(tree_shrew_lengths[chrom] for chrom in fold_data['train'])
        total_bases = test_bases + valid_bases + train_bases
        
        results = {
            'test_bases': test_bases,
            'valid_bases': valid_bases, 
            'train_bases': train_bases,
            'total_bases': total_bases,
            'test_percent': test_bases / total_bases * 100,
            'valid_percent': valid_bases / total_bases * 100,
            'train_percent': train_bases / total_bases * 100
        }
        all_results[fold_name] = results
        
        # 统计染色体使用情况
        for chrom in fold_data['test']:
            chrom_usage[chrom]['test'] += 1
        for chrom in fold_data['valid']:
            chrom_usage[chrom]['valid'] += 1
        for chrom in fold_data['train']:
            chrom_usage[chrom]['train'] += 1
        
        print(f"=== {fold_name} ===")
        print(f"训练集: {len(fold_data['train'])}条染色体, {train_bases:,} bp ({results['train_percent']:.2f}%)")
        print(f"测试集: {len(fold_data['test'])}条染色体, {test_bases:,} bp ({results['test_percent']:.2f}%)")
        print(f"验证集: {len(fold_data['valid'])}条染色体, {valid_bases:,} bp ({results['valid_percent']:.2f}%)")
        print(f"测试染色体: {fold_data['test']}")
        print(f"验证染色体: {fold_data['valid']}")
        print()
    
    # 汇总统计
    print("=== 汇总统计 ===")
    print(f"{'Fold':<8} {'训练集(%)':<12} {'测试集(%)':<12} {'验证集(%)':<12} {'训练染色体':<15} {'测试染色体':<15} {'验证染色体':<15}")
    print("-" * 100)
    
    for fold_name in folds.keys():
        r = all_results[fold_name]
        fold_data = folds[fold_name]
        print(f"{fold_name:<8} {r['train_percent']:<11.2f} {r['test_percent']:<11.2f} {r['valid_percent']:<11.2f} "
              f"{len(fold_data['train']):<14} {len(fold_data['test']):<14} {len(fold_data['valid']):<14}")
    
    # 计算平均值
    avg_train = sum(r['train_percent'] for r in all_results.values()) / 5
    avg_test = sum(r['test_percent'] for r in all_results.values()) / 5
    avg_valid = sum(r['valid_percent'] for r in all_results.values()) / 5
    
    print("-" * 100)
    print(f"{'平均':<8} {avg_train:<11.2f} {avg_test:<11.2f} {avg_valid:<11.2f}")
    
    # 验证染色体覆盖
    print("\n=== 染色体覆盖统计 ===")
    for chrom, usage in chrom_usage.items():
        total_uses = usage['test'] + usage['valid'] + usage['train']
        print(f"{chrom}: 测试{usage['test']}次, 验证{usage['valid']}次, 训练{usage['train']}次")
    
    # 检查每个染色体在训练+验证中的出现次数
    print("\n=== 训练+验证覆盖检查 ===")
    all_covered = True
    for chrom, usage in chrom_usage.items():
        train_valid_uses = usage['train'] + usage['valid']
        if train_valid_uses < 4:  # 在5个fold中至少4次出现在训练或验证集
            print(f"⚠️  {chrom}: 训练+验证中只出现{train_valid_uses}次")
            all_covered = False
        else:
            print(f"✅  {chrom}: 训练+验证中出现{train_valid_uses}次")
    
    if all_covered:
        print("\n🎯 所有染色体在训练集或验证集中都有充分覆盖!")
    
    return all_results, chrom_usage

In [ ]:
# 创建并验证划分
best_fold = create_optimized_tree_shrew_folds_v2(max_iterations=1000,seed=0)
results, chrom_usage = calculate_and_validate_folds(best_fold)

In [ ]:
# 保存划分方案
import json

output = {
    'total_bases': sum(tree_shrew_lengths.values()),
    'target_ratio': {'train': 0.7, 'test': 0.2, 'valid': 0.1},
    'chromosome_lengths': tree_shrew_lengths,
    'folds': best_fold
}

with open('/home/oulin/chrombpnet/chrombpnet_TS_Mono/data/tree_shrew_random_7_2_1_split.json', 'w') as f:
    json.dump(output, f, indent=2)